# Data Cleaning - Olist E-commerce Dataset


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
customers = pd.read_csv("olist_customers_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
order_reviews = pd.read_csv("olist_order_reviews_dataset.csv")
order_payments = pd.read_csv("olist_order_payments_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
product_names = pd.read_csv("product_category_name_translation.csv")

print("customers:", customers.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("order_reviews:", order_reviews.shape)
print("order_payments:", order_payments.shape)
print("products:", products.shape)

customers: (99441, 5)
orders: (99441, 8)
order_items: (112650, 7)
order_reviews: (99224, 7)
order_payments: (103886, 5)
products: (32951, 9)


In [3]:
print("orders - nulls:", orders.isnull().sum().sum(), "duplicates:", orders.duplicated().sum())
print("order_items - nulls:", order_items.isnull().sum().sum(), "duplicates:", order_items.duplicated().sum())
print("order_payments - nulls:", order_payments.isnull().sum().sum(), "duplicates:", order_payments.duplicated().sum())

orders - nulls: 4908 duplicates: 0
order_items - nulls: 0 duplicates: 0
order_payments - nulls: 0 duplicates: 0


## Merge Datasets

In [24]:
payments_agg = order_payments.groupby("order_id", as_index=False).agg(
    payment_value=("payment_value", "sum"),
    payment_installments=("payment_installments", "mean"),
    payment_type=("payment_type", "first"),
)

df = (
    orders
    .merge(order_items, on="order_id", how="left")
    .merge(order_reviews, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(customers, on="customer_id", how="left")
    .merge(products, on="product_id", how="left")
    .merge(product_names, on="product_category_name", how="left")
)

print("Merged shape:", df.shape)
print("Duplicate order+item rows:", df.duplicated(subset=["order_id", "order_item_id"]).sum())
df.info()

Merged shape: (114092, 36)
Duplicate order+item rows: 667
<class 'pandas.DataFrame'>
RangeIndex: 114092 entries, 0 to 114091
Data columns (total 36 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       114092 non-null  str    
 1   customer_id                    114092 non-null  str    
 2   order_status                   114092 non-null  str    
 3   order_purchase_timestamp       114092 non-null  str    
 4   order_approved_at              113930 non-null  str    
 5   order_delivered_carrier_date   112112 non-null  str    
 6   order_delivered_customer_date  110839 non-null  str    
 7   order_estimated_delivery_date  114092 non-null  str    
 8   order_item_id                  113314 non-null  float64
 9   product_id                     113314 non-null  str    
 10  seller_id                      113314 non-null  str    
 11  shipping_limit_date            113314 non-nu

In [23]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_limit_date",
    "review_creation_date",
    "review_answer_timestamp",
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df.info()

<class 'pandas.DataFrame'>
Index: 110813 entries, 0 to 114091
Data columns (total 37 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       110813 non-null  str           
 1   customer_id                    110813 non-null  str           
 2   order_status                   110813 non-null  str           
 3   order_purchase_timestamp       110813 non-null  datetime64[us]
 4   order_approved_at              110813 non-null  datetime64[us]
 5   order_delivered_carrier_date   110813 non-null  datetime64[us]
 6   order_delivered_customer_date  110813 non-null  datetime64[us]
 7   order_estimated_delivery_date  110813 non-null  datetime64[us]
 8   order_item_id                  110813 non-null  float64       
 9   product_id                     110813 non-null  str           
 10  seller_id                      110813 non-null  str           
 11  shipping_limit_d

In [25]:
df.isnull().sum()/len(df) * 100

order_id                          0.000000
customer_id                       0.000000
order_status                      0.000000
order_purchase_timestamp          0.000000
order_approved_at                 0.141991
order_delivered_carrier_date      1.735442
order_delivered_customer_date     2.851208
order_estimated_delivery_date     0.000000
order_item_id                     0.681906
product_id                        0.681906
seller_id                         0.681906
shipping_limit_date               0.681906
price                             0.681906
freight_value                     0.681906
review_id                         0.842303
review_score                      0.842303
review_comment_title             88.147285
review_comment_message           57.783193
review_creation_date              0.842303
review_answer_timestamp           0.842303
payment_value                     0.002629
payment_installments              0.002629
payment_type                      0.002629
customer_un

## Handle Missing Values and Imputation

In [6]:
drop_cols = [
    "review_comment_message",
    "review_comment_title",
    "review_id",
    "product_category_name",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
]
df = df.drop(columns=drop_cols)
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,review_score,review_creation_date,review_answer_timestamp,payment_value,payment_installments,payment_type,customer_unique_id,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,4.0,2017-10-11,2017-10-12 03:43:48,38.71,1.0,credit_card,7c396fd4830fd04220f754e42b4e5bff,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,4.0,2018-08-08,2018-08-08 18:37:50,141.46,1.0,boleto,af07308b275d755c9edb36a90c618231,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,5.0,2018-08-18,2018-08-22 19:07:58,179.12,3.0,credit_card,3a653a41f6f9fc3d2a113cf8398680e8,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,5.0,2017-12-03,2017-12-05 19:21:58,72.20,1.0,credit_card,7c142cf63193a1473d2e66489a9ae977,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,5.0,2018-02-17,2018-02-18 13:02:51,28.62,1.0,credit_card,72632f0f9dd73dfee390c9b22eb56dd6,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery


In [7]:
df = df[df["order_status"] == "delivered"].copy()

df.dropna(
    subset=[
        "payment_value",
        "order_item_id",
        "product_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "shipping_limit_date",
    ],
    inplace=True,
)

print("Shape after cleaning:", df.shape)

Shape after cleaning: (110813, 29)


In [8]:
num_cols = [
    "price", "freight_value", "product_weight_g", "product_length_cm",
    "product_height_cm", "product_width_cm", "product_name_lenght",
    "product_description_lenght", "product_photos_qty", "review_score",
]

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

df["product_category_name_english"] = df["product_category_name_english"].fillna("unknown")
df.isnull().sum().sort_values(ascending=False).head(10)

review_answer_timestamp          827
review_creation_date             827
order_status                       0
customer_id                        0
order_id                           0
order_delivered_carrier_date       0
order_delivered_customer_date      0
order_estimated_delivery_date      0
order_item_id                      0
product_id                         0
dtype: int64

## Feature Engineering

In [9]:
df["approval_time_hours"] = (df["order_approved_at"] - df["order_purchase_timestamp"]).dt.total_seconds() / 3600
df["delivery_time_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days
df["delivery_delay_days"] = (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.days
df["shipping_delay_days"] = (df["order_delivered_carrier_date"] - df["shipping_limit_date"]).dt.days
df["review_response_days"] = (df["review_answer_timestamp"] - df["review_creation_date"]).dt.days
df["purchase_year"] = df["order_purchase_timestamp"].dt.year
df["purchase_month"] = df["order_purchase_timestamp"].dt.month
df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour
df[["approval_time_hours", "delivery_time_days", "delivery_delay_days"]].head()

,approval_time_hours,delivery_time_days,delivery_delay_days
0,0.178333,8,-8
1,30.713889,13,-6
2,0.276111,9,-18
3,0.298056,13,-13
4,1.030556,2,-10


## Repeat Purchase Target (Classification)

In [10]:
order_dates = (
    df[["customer_unique_id", "order_id", "order_purchase_timestamp"]]
    .drop_duplicates(subset=["customer_unique_id", "order_id"])
    .sort_values(["customer_unique_id", "order_purchase_timestamp"]))

order_dates["next_purchase_date"] = (
    order_dates.groupby("customer_unique_id")["order_purchase_timestamp"].shift(-1))

order_dates["days_to_next_purchase"] = (
    order_dates["next_purchase_date"] - order_dates["order_purchase_timestamp"]).dt.days

order_dates["Repeat_Purchase"] = (
    (order_dates["days_to_next_purchase"] <= 90) & order_dates["days_to_next_purchase"].notna()).astype(int)

customer_target = (
    order_dates.groupby("customer_unique_id")["Repeat_Purchase"].max().reset_index())

print(customer_target["Repeat_Purchase"].value_counts())
order_dates.head()

Repeat_Purchase
0    91377
1     1958
Name: count, dtype: int64


,customer_unique_id,order_id,order_purchase_timestamp,next_purchase_date,days_to_next_purchase,Repeat_Purchase
60486,0000366f3b9a7992bf8c76cfdf3221e2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27,NaT,NaN,0
84688,0000b849f77a49e4a4ce2b2a4ca5be3f,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27,NaT,NaN,0
30316,0000f46a3911fa3c0805444483337064,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03,NaT,NaN,0
112977,0000f6ccb0745a6a4b88665a16c9f078,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41,NaT,NaN,0
47622,0004aac84e0df4da2b147fca70cf8255,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42,NaT,NaN,0


## Customer Dataset (Classification)

In [11]:
order_level = df.groupby(["customer_unique_id", "order_id"], as_index=False).agg(
    order_purchase_timestamp=("order_purchase_timestamp", "first"),
    price=("price", "sum"),
    freight_value=("freight_value", "sum"),
    payment_value=("payment_value", "first"),
    payment_installments=("payment_installments", "mean"),
    review_score=("review_score", "mean"),
    product_photos_qty=("product_photos_qty", "mean"),
    product_weight_g=("product_weight_g", "mean"),
    approval_time_hours=("approval_time_hours", "mean"),
    delivery_time_days=("delivery_time_days", "mean"),
    delivery_delay_days=("delivery_delay_days", "mean"),
    shipping_delay_days=("shipping_delay_days", "mean"),
    review_response_days=("review_response_days", "mean"),
    purchase_year=("purchase_year", "first"),
    purchase_month=("purchase_month", "first"),
    purchase_hour=("purchase_hour", "mean"),
)

first_orders = (
    order_level.sort_values("order_purchase_timestamp")
    .groupby("customer_unique_id", as_index=False)
    .first()
)

customer_df = pd.DataFrame()
customer_df["customer_unique_id"] = first_orders["customer_unique_id"]
customer_df["total_orders"] = 1
customer_df["total_price"] = first_orders["price"]
customer_df["average_price"] = first_orders["price"]
customer_df["average_freight"] = first_orders["freight_value"]
customer_df["average_review"] = first_orders["review_score"]
customer_df["average_installments"] = first_orders["payment_installments"]
customer_df["total_payment"] = first_orders["payment_value"]
customer_df["average_photos"] = first_orders["product_photos_qty"]
customer_df["average_weight"] = first_orders["product_weight_g"]
customer_df["approval_time"] = first_orders["approval_time_hours"]
customer_df["delivery_time"] = first_orders["delivery_time_days"]
customer_df["delivery_delay"] = first_orders["delivery_delay_days"]
customer_df["shipping_delay"] = first_orders["shipping_delay_days"]
customer_df["review_response"] = first_orders["review_response_days"]
customer_df["purchase_year"] = first_orders["purchase_year"]
customer_df["purchase_month"] = first_orders["purchase_month"]
customer_df["purchase_hour"] = first_orders["purchase_hour"]

customer_df = customer_df.merge(customer_target, on="customer_unique_id", how="left")

customer_df["review_response"] = customer_df["review_response"].fillna(customer_df["review_response"].median())

print("customer_df shape:", customer_df.shape)
print("Repeat_Purchase rate:", round(customer_df["Repeat_Purchase"].mean(), 4))
print("Missing values:", customer_df.isnull().sum().sum())
customer_df.head()

customer_df shape: (93335, 19)
Repeat_Purchase rate: 0.021
Missing values: 0


,customer_unique_id,total_orders,total_price,average_price,average_freight,average_review,average_installments,total_payment,average_photos,average_weight,approval_time,delivery_time,delivery_delay,shipping_delay,review_response,purchase_year,purchase_month,purchase_hour,Repeat_Purchase
0,0000366f3b9a7992bf8c76cfdf3221e2,1,129.90,129.90,12.00,5.0,8.0,141.90,1.0,1500.0,0.247500,6.0,-5.0,-4.0,4.0,2018,5,10.0,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,18.90,18.90,8.29,4.0,1.0,27.19,1.0,375.0,7.238056,3.0,-5.0,-3.0,0.0,2018,5,11.0,0
2,0000f46a3911fa3c0805444483337064,1,69.00,69.00,17.22,3.0,8.0,86.22,3.0,1500.0,0.000000,25.0,-2.0,-3.0,1.0,2017,3,21.0,0
3,0000f6ccb0745a6a4b88665a16c9f078,1,25.99,25.99,17.63,4.0,4.0,43.62,5.0,150.0,0.326667,20.0,-12.0,-6.0,1.0,2017,10,20.0,0
4,0004aac84e0df4da2b147fca70cf8255,1,180.00,180.00,16.89,5.0,6.0,196.89,3.0,6050.0,0.352778,13.0,-8.0,-7.0,4.0,2017,11,19.0,0


In [12]:
customer_df.to_csv("customer_df.csv", index=False)
print("Saved customer_df.csv")

Saved customer_df.csv


## Revenue Dataset

#### past/future split by 70% cutoff, order-level payment sums, tenure in days.

In [13]:
cutoff_date = df["order_purchase_timestamp"].quantile(0.70)
print("Cutoff date:", cutoff_date)

past_orders = df[df["order_purchase_timestamp"] < cutoff_date].copy()
future_orders = df[df["order_purchase_timestamp"] >= cutoff_date].copy()

print("Past orders:", past_orders.shape)
print("Future orders:", future_orders.shape)

Cutoff date: 2018-04-15 12:14:23
Past orders: (77568, 37)
Future orders: (33245, 37)


In [14]:
past_order_level = past_orders.groupby(["customer_unique_id", "order_id"], as_index=False).agg(
    order_purchase_timestamp=("order_purchase_timestamp", "first"),
    price=("price", "sum"),
    payment_value=("payment_value", "first"),
    payment_installments=("payment_installments", "mean"),
    review_score=("review_score", "mean"),
    purchase_year=("purchase_year", "first"),
    purchase_month=("purchase_month", "first"),
)

group = past_order_level.groupby("customer_unique_id")

revenue_df = pd.DataFrame()
revenue_df["customer_unique_id"] = group.size().index
revenue_df["total_orders"] = group["order_id"].nunique().values
revenue_df["total_price"] = group["price"].sum().values
revenue_df["last_purchase_year"] = group["purchase_year"].max().values
revenue_df["last_purchase_month"] = group["purchase_month"].max().values
revenue_df["first_purchase_year"] = group["purchase_year"].min().values

first_date = past_order_level.groupby("customer_unique_id")["order_purchase_timestamp"].min()
last_date = past_order_level.groupby("customer_unique_id")["order_purchase_timestamp"].max()
revenue_df["customer_tenure_days"] = (last_date - first_date).dt.days.values

revenue_df["average_installments"] = group["payment_installments"].mean().values
revenue_df["average_review"] = group["review_score"].mean().values
revenue_df["past_revenue"] = group["payment_value"].sum().values
revenue_df["avg_order_value"] = revenue_df["past_revenue"] / revenue_df["total_orders"]

future_order_level = future_orders.groupby(["customer_unique_id", "order_id"], as_index=False).agg(
    payment_value=("payment_value", "first")
)
future_revenue = future_order_level.groupby("customer_unique_id")["payment_value"].sum()
revenue_df["future_revenue"] = revenue_df["customer_unique_id"].map(future_revenue).fillna(0)

print("revenue_df shape:", revenue_df.shape)
revenue_df.head()

revenue_df shape: (65277, 12)


,customer_unique_id,total_orders,total_price,last_purchase_year,last_purchase_month,first_purchase_year,customer_tenure_days,average_installments,average_review,past_revenue,avg_order_value,future_revenue
0,0000f46a3911fa3c0805444483337064,1,69.00,2017,3,2017,0,8.0,3.0,86.22,86.22,0.0
1,0000f6ccb0745a6a4b88665a16c9f078,1,25.99,2017,10,2017,0,4.0,4.0,43.62,43.62,0.0
2,0004aac84e0df4da2b147fca70cf8255,1,180.00,2017,11,2017,0,6.0,5.0,196.89,196.89,0.0
3,0004bd2a26a76fe21f786e4fbd80607f,1,154.00,2018,4,2018,0,8.0,4.0,166.98,166.98,0.0
4,00053a61a98854899e70ed204dd4bafe,1,382.00,2018,2,2018,0,3.0,1.0,419.18,419.18,0.0


In [15]:
revenue_df.to_csv("revenue_dataset.csv", index=False)
print("Saved revenue_dataset.csv")

Saved revenue_dataset.csv


In [34]:
revenue_df["future_revenue"].unique()

array([   0.  ,   80.32,  340.07,  157.86,  271.64,  234.13,   33.33,
         71.78,  320.58,   31.31,   37.85,   81.24,  318.61,  248.28,
        252.8 ,   67.17,  701.43,  190.46,  620.15,   68.37,   35.59,
        169.74,  180.3 ,   92.97,  133.06,  196.34,  126.57,   93.28,
         62.65,   22.58,   27.46,   64.66,   95.8 ,  210.08,  385.74,
        147.48,  119.79,   89.48,  124.41,  193.98,   87.18,  128.67,
        186.1 ,  249.31,  135.94,   57.79,   88.71,   74.88,  150.84,
         37.36,  138.62,  181.69,  138.39,  643.34,   99.65,   62.78,
        149.11,   61.38,   98.56,  146.77,  309.99,   47.04,   88.45,
         64.44,  141.13,  230.  ,  136.91,   62.69,   64.22,  207.62,
         91.23,  210.14,   75.68,   72.08,  274.56,  162.33,  138.65,
        356.13,  228.87,  228.86,   80.4 ,   37.24,  245.11, 1402.32,
        162.3 ,  254.94,   98.26,  218.51,   21.38,  114.77,   40.25,
        152.95,   46.22,  100.22,  117.49,   96.26,  175.72,  223.08,
         89.03,  130

## Data Quality Summary

In [16]:
print("=== Final Summary ===")
print("Classification rows:", len(customer_df))
print("Repeat purchase rate:", round(customer_df["Repeat_Purchase"].mean() * 100, 2), "%")
print("Revenue rows:", len(revenue_df))
print("Future revenue > 0:", round((revenue_df["future_revenue"] > 0).mean() * 100, 2), "%")
print("Missing values in customer_df:", customer_df.isnull().sum().sum())
print("Missing values in revenue_df:", revenue_df.isnull().sum().sum())

=== Final Summary ===
Classification rows: 93335
Repeat purchase rate: 2.1 %
Revenue rows: 65277
Future revenue > 0: 0.87 %
Missing values in customer_df: 0
Missing values in revenue_df: 0


In [17]:
!pip3 install xgboost



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import xgboost
print(xgboost.__version__)


3.4.0
